# Wound Segmentation — Mask R-CNN (Kaggle T4)

## 1. Config

In [ ]:
import os
import torch
import torchvision

# ===== PATHS =====
ROOT          = "/kaggle/input/datasets/seoyeongb/wound-seg-data/data_wound_seg"
TRAIN_IMG_DIR = os.path.join(ROOT, "train_images")
TRAIN_MSK_DIR = os.path.join(ROOT, "train_masks")
TEST_IMG_DIR  = os.path.join(ROOT, "test_images")
TEST_MSK_DIR  = os.path.join(ROOT, "test_masks")
OUT_DIR       = "/kaggle/working/out"
MODEL_SAVE_PATH = "/kaggle/working/mask_rcnn_wound.pth"

# ===== MODEL =====
NUM_CLASSES    = 2        # 0: background, 1: wound
PRETRAINED     = True
FREEZE_BACKBONE = True    # backbone 고정 -> 빠른 학습, 과적합 방지

# ===== TRAINING =====
TRAIN_SAMPLE_SIZE = 500   # 스마트 샘플링 장수
EPOCHS        = 10
BATCH_SIZE    = 2
LR            = 1e-4
WEIGHT_DECAY  = 1e-4
RESIZE_TO     = 512

# ===== INFERENCE =====
SCORE_THR     = 0.5
MASK_THR      = 0.5
MAX_INSTANCES = 5         # 이미지 한 장당 최대 검출 인스턴스 수 (다중 상처 대응)
NUM_TEST_SAMPLES = 100
INFER_OUT_DIR = os.path.join(OUT_DIR, "infer_kaggle_v1")

# ===== DEVICE =====
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(INFER_OUT_DIR, exist_ok=True)

print(f"device      : {device}")
print(f"torch       : {torch.__version__}")
print(f"torchvision : {torchvision.__version__}")
print(f"ROOT exists : {os.path.isdir(ROOT)}")

## 2. 스마트 500장 샘플링

마스크 면적을 직접 계산한 뒤, 작은/중간/큰 상처 고르게 500장 선택

In [ ]:
import glob
import cv2
import numpy as np

def compute_mask_area(mask_path: str) -> int:
    """
    마스크 이미지에서 양성(>0) 픽셀 수를 반환.

    Args:
        mask_path (str): 마스크 파일 경로
    Returns:
        int: 픽셀 면적 (0이면 빈 마스크)
    """
    m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if m is None:
        return 0
    return int((m > 0).sum())

def smart_sample(img_dir: str, mask_dir: str, n: int, seed: int = 42):
    """
    마스크 면적 기준으로 균등하게 n장 샘플링.
    small 30% / mid 40% / large 30% 비율로 선택.

    Args:
        img_dir (str): 이미지 디렉토리
        mask_dir (str): 마스크 디렉토리
        n (int): 샘플링할 총 이미지 수
        seed (int): 랜덤 시드
    Returns:
        list[str]: 선택된 파일명 리스트
    """
    np.random.seed(seed)

    all_fns = sorted([
        os.path.basename(p)
        for p in glob.glob(os.path.join(img_dir, "*"))
        if p.lower().endswith((".png", ".jpg", ".jpeg"))
    ])

    # 이미지 & 마스크 모두 존재하고 마스크가 비어 있지 않은 것만 수집
    valid = []
    for fn in all_fns:
        msk_path = os.path.join(mask_dir, fn)
        if not os.path.exists(msk_path):
            continue
        area = compute_mask_area(msk_path)
        if area > 0:
            valid.append((fn, area))

    valid.sort(key=lambda x: x[1])  # 면적 오름차순 정렬
    total = len(valid)
    print(f"유효 샘플 수: {total}")

    n = min(n, total)
    k_small = int(round(n * 0.30))
    k_mid   = int(round(n * 0.40))
    k_large = n - k_small - k_mid

    small_pool = valid[:total // 3]
    mid_pool   = valid[total // 3 : 2 * total // 3]
    large_pool = valid[2 * total // 3:]

    def pick(pool, k):
        fns = [fn for fn, _ in pool]
        return list(np.random.choice(fns, size=min(k, len(fns)), replace=False))

    selected = pick(small_pool, k_small) + pick(mid_pool, k_mid) + pick(large_pool, k_large)

    # 중복 제거 후 n개 맞춤
    selected = list(dict.fromkeys(selected))[:n]
    print(f"선택된 샘플 수: {len(selected)} (small={k_small}, mid={k_mid}, large={k_large})")
    return selected


sampled_files = smart_sample(TRAIN_IMG_DIR, TRAIN_MSK_DIR, TRAIN_SAMPLE_SIZE)

## 3. Dataset

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class WoundSegDataset(Dataset):
    """
    Mask R-CNN 학습용 상처 세그멘테이션 데이터셋.

    Args:
        img_dir (str): 이미지 디렉토리 경로
        mask_dir (str): 마스크 디렉토리 경로
        file_list (list[str] | None): 사용할 파일명 리스트. None이면 전체 사용.
        resize_to (int): 리사이즈 크기 (정사각형)
    """

    def __init__(self, img_dir: str, mask_dir: str,
                 file_list=None, resize_to: int = 512):
        self.img_dir  = img_dir
        self.mask_dir = mask_dir
        self.resize_to = resize_to

        if file_list is None:
            self.files = sorted([
                os.path.basename(p)
                for p in glob.glob(os.path.join(img_dir, "*"))
            ])
        else:
            self.files = list(file_list)

        # 이미지/마스크 둘 다 존재하는 것만 유지
        self.files = [
            fn for fn in self.files
            if os.path.exists(os.path.join(img_dir, fn))
            and os.path.exists(os.path.join(mask_dir, fn))
        ]

    def __len__(self) -> int:
        return len(self.files)

    def __getitem__(self, idx: int):
        fn = self.files[idx]

        # 이미지: BGR -> RGB, float32 [C,H,W] 0~1
        bgr = cv2.imread(os.path.join(self.img_dir, fn), cv2.IMREAD_COLOR)
        if bgr is None:
            raise FileNotFoundError(os.path.join(self.img_dir, fn))
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        rgb = cv2.resize(rgb, (self.resize_to, self.resize_to))
        img = torch.from_numpy(rgb).permute(2, 0, 1).float() / 255.0

        # 마스크: grayscale -> binary uint8
        m = cv2.imread(os.path.join(self.mask_dir, fn), cv2.IMREAD_GRAYSCALE)
        if m is None:
            raise FileNotFoundError(os.path.join(self.mask_dir, fn))
        mask = (m > 127).astype(np.uint8)
        mask = cv2.resize(mask, (self.resize_to, self.resize_to),
                          interpolation=cv2.INTER_NEAREST)

        ys, xs = np.where(mask > 0)
        if len(xs) == 0 or len(ys) == 0:
            # 빈 마스크면 다음 샘플로 교체
            return self.__getitem__((idx + 1) % len(self.files))

        x1, x2 = int(xs.min()), int(xs.max())
        y1, y2 = int(ys.min()), int(ys.max())

        boxes  = torch.tensor([[x1, y1, x2, y2]], dtype=torch.float32)
        labels = torch.tensor([1], dtype=torch.int64)
        masks  = torch.from_numpy(mask[None, :, :])
        area   = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])

        target = {
            "boxes"   : boxes,
            "labels"  : labels,
            "masks"   : masks,
            "image_id": torch.tensor([idx]),
            "area"    : area,
            "iscrowd" : torch.zeros((1,), dtype=torch.int64),
        }
        return img, target, fn


def collate_fn(batch):
    imgs, targets, fns = zip(*batch)
    return list(imgs), list(targets), list(fns)


train_ds = WoundSegDataset(TRAIN_IMG_DIR, TRAIN_MSK_DIR,
                           file_list=sampled_files, resize_to=RESIZE_TO)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=2,
                          collate_fn=collate_fn)

print(f"학습 데이터셋 크기: {len(train_ds)}")

## 4. 모델 생성 (Backbone Freeze)

In [ ]:
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

def build_model(num_classes: int, pretrained: bool = True,
                freeze_backbone: bool = True):
    """
    Mask R-CNN ResNet-50 FPN 모델을 구성.
    Backbone을 freeze해 빠른 학습과 과적합 방지.

    Args:
        num_classes (int): 배경 포함 클래스 수 (wound=2)
        pretrained (bool): ImageNet 사전학습 가중치 사용 여부
        freeze_backbone (bool): backbone 레이어 고정 여부
    Returns:
        torch.nn.Module: 구성된 Mask R-CNN 모델
    """
    weights = "DEFAULT" if pretrained else None
    model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights=weights)

    # Backbone 고정: body(ResNet-50) + FPN 파라미터를 모두 동결
    if freeze_backbone:
        for name, param in model.named_parameters():
            if "backbone" in name:
                param.requires_grad = False
        frozen = sum(1 for n, p in model.named_parameters()
                     if "backbone" in n and not p.requires_grad)
        print(f"Backbone 동결 파라미터 수: {frozen}")

    # Box predictor 교체
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    # Mask predictor 교체
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(
        in_features_mask, 256, num_classes
    )

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"학습 파라미터: {trainable:,} / 전체: {total:,}")
    return model


model = build_model(NUM_CLASSES, PRETRAINED, FREEZE_BACKBONE)
model = model.to(device)

## 5. 학습 루프

In [ ]:
import torch.optim as optim

# requires_grad=True인 파라미터만 옵티마이저에 등록
params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)


def train_one_epoch(model, loader, optimizer, epoch: int) -> float:
    """
    1 epoch 학습을 수행하고 평균 손실을 반환.

    Args:
        model: Mask R-CNN 모델
        loader: DataLoader
        optimizer: AdamW 옵티마이저
        epoch (int): 현재 에폭 번호 (로그 출력용)
    Returns:
        float: 에폭 평균 손실
    """
    model.train()
    total_loss = 0.0

    for step, (imgs, targets, _) in enumerate(loader):
        imgs    = [img.to(device) for img in imgs]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(imgs, targets)
        loss = sum(loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if (step + 1) % 50 == 0:
            detail = {k: f"{v.item():.4f}" for k, v in loss_dict.items()}
            print(f"  epoch {epoch} step {step+1}/{len(loader)} | "
                  f"loss={loss.item():.4f} | {detail}")

    return total_loss / max(1, len(loader))


loss_history = []

for epoch in range(1, EPOCHS + 1):
    avg_loss = train_one_epoch(model, train_loader, optimizer, epoch)
    scheduler.step()
    loss_history.append(avg_loss)
    print(f"[Epoch {epoch:02d}/{EPOCHS}] avg_loss={avg_loss:.4f} "
          f"lr={scheduler.get_last_lr()[0]:.2e}")

print("학습 완료")

## 6. Loss 곡선 저장 & 모델 저장

In [ ]:
import matplotlib.pyplot as plt

# Loss 곡선
plt.figure()
plt.plot(range(1, EPOCHS + 1), loss_history, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Avg Loss")
plt.title("Training Loss Curve")
plt.tight_layout()
loss_plot_path = os.path.join(OUT_DIR, "train_loss_curve.png")
plt.savefig(loss_plot_path, dpi=150)
plt.show()
print("Loss 곡선 저장 ->", loss_plot_path)

# 모델 저장
torch.save(model.state_dict(), MODEL_SAVE_PATH)
print("모델 저장 ->", MODEL_SAVE_PATH)

## 7. 추론 & 평가

In [ ]:
import pandas as pd
import time

# ---- 헬퍼 함수 ----

def read_gt_bin(mask_path: str, target_size=None) -> np.ndarray | None:
    """
    GT 마스크를 읽어 binary numpy 배열로 반환. (BGR 아닌 GRAY)

    Args:
        mask_path (str): 마스크 파일 경로
        target_size (tuple | None): (W, H) 리사이즈 크기
    Returns:
        np.ndarray | None: uint8 binary 마스크 or None
    """
    gt = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if gt is None:
        return None
    if target_size is not None:
        gt = cv2.resize(gt, target_size, interpolation=cv2.INTER_NEAREST)
    return (gt > 0).astype(np.uint8)


def postprocess_mask(bin_mask: np.ndarray,
                     k_close: int = 7, k_open: int = 3,
                     min_area: int = 200) -> np.ndarray:
    """
    Morphological closing/opening + 소형 컴포넌트 제거.

    Args:
        bin_mask (np.ndarray): uint8 binary 마스크
        k_close (int): closing 커널 크기
        k_open (int): opening 커널 크기
        min_area (int): 이 픽셀 수 미만의 컴포넌트는 제거
    Returns:
        np.ndarray: 후처리된 binary 마스크
    """
    m = (bin_mask * 255).astype(np.uint8)
    kernel_c = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_close, k_close))
    kernel_o = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_open, k_open))
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, kernel_c)
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN,  kernel_o)
    m = (m > 0).astype(np.uint8)

    if min_area > 0:
        num, labels, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
        out = np.zeros_like(m)
        for i in range(1, num):
            if stats[i, cv2.CC_STAT_AREA] >= min_area:
                out[labels == i] = 1
        m = out
    return m


def compute_metrics(pred_bin: np.ndarray, gt_bin: np.ndarray):
    """
    IoU / Dice / Precision / Recall 계산.

    Args:
        pred_bin (np.ndarray): 예측 binary 마스크
        gt_bin (np.ndarray): GT binary 마스크
    Returns:
        tuple: (iou, dice, precision, recall, tp, fp, fn)
    """
    pred = pred_bin.astype(bool)
    gt   = gt_bin.astype(bool)
    tp = int(np.logical_and(pred, gt).sum())
    fp = int(np.logical_and(pred, ~gt).sum())
    fn = int(np.logical_and(~pred, gt).sum())

    iou  = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else None
    dice = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else None
    prec = tp / (tp + fp) if (tp + fp) > 0 else None
    rec  = tp / (tp + fn) if (tp + fn) > 0 else None
    return iou, dice, prec, rec, tp, fp, fn


def make_overlay(bgr: np.ndarray, pred_bin: np.ndarray,
                 gt_bin: np.ndarray, alpha: float = 0.55) -> np.ndarray:
    """
    GT(green) / Pred(red) / Overlap(yellow) 오버레이 이미지 생성.
    입력은 BGR 이미지.

    Args:
        bgr (np.ndarray): 원본 BGR 이미지
        pred_bin (np.ndarray): 예측 binary 마스크
        gt_bin (np.ndarray): GT binary 마스크
        alpha (float): 오버레이 투명도
    Returns:
        np.ndarray: 오버레이된 BGR 이미지
    """
    h, w = bgr.shape[:2]
    overlay = bgr.copy()
    gt, pr  = gt_bin.astype(bool), pred_bin.astype(bool)
    overlap = gt & pr
    gt_only = gt & ~pr
    pr_only = pr & ~gt

    color = np.zeros((h, w, 3), dtype=np.uint8)
    color[gt_only] = (0, 255, 0)    # green
    color[pr_only] = (0, 0, 255)    # red
    color[overlap] = (0, 255, 255)  # yellow

    mask_any = gt_only | pr_only | overlap
    overlay[mask_any] = (
        overlay[mask_any] * (1 - alpha) + color[mask_any] * alpha
    ).astype(np.uint8)
    return overlay


def make_multi_instance_overlay(bgr: np.ndarray,
                                instance_masks: list,
                                alpha: float = 0.6) -> np.ndarray:
    """
    각 인스턴스를 고유 색상으로 오버레이. Mask R-CNN의 인스턴스 분리 능력을 시각화.

    U-Net(Semantic Segmentation)은 모든 상처를 단일 덩어리로 합치지만,
    Mask R-CNN은 각 상처에 고유 ID를 부여하여 독립적으로 추적 가능.

    Args:
        bgr (np.ndarray): 원본 BGR 이미지
        instance_masks (list[np.ndarray]): 인스턴스별 binary 마스크 리스트
        alpha (float): 오버레이 투명도
    Returns:
        np.ndarray: 인스턴스별 색상으로 오버레이된 BGR 이미지
    """
    # 인스턴스별 고유 색상 (BGR)
    INSTANCE_COLORS = [
        (0,   0,   255),   # #1 빨강
        (255, 0,   0  ),   # #2 파랑
        (0,   200, 0  ),   # #3 초록
        (0,   200, 200),   # #4 노랑
        (200, 0,   200),   # #5 마젠타
    ]

    overlay = bgr.copy()
    h, w = bgr.shape[:2]

    for i, mask in enumerate(instance_masks):
        color = INSTANCE_COLORS[i % len(INSTANCE_COLORS)]
        color_layer = np.zeros((h, w, 3), dtype=np.uint8)
        region = mask.astype(bool)
        color_layer[region] = color
        overlay[region] = (
            overlay[region] * (1 - alpha) + color_layer[region] * alpha
        ).astype(np.uint8)

        # 인스턴스 번호 레이블
        ys, xs = np.where(region)
        if len(xs) > 0:
            cx, cy = int(xs.mean()), int(ys.mean())
            cv2.putText(overlay, f"#{i+1}", (cx - 10, cy),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2,
                        cv2.LINE_AA)

    return overlay


def make_merged_overlay(bgr: np.ndarray,
                        pred_bin: np.ndarray,
                        alpha: float = 0.6) -> np.ndarray:
    """
    모든 인스턴스를 단일 색(빨강)으로 합쳐서 오버레이.
    Semantic Segmentation(U-Net) 방식의 출력을 시뮬레이션.

    Args:
        bgr (np.ndarray): 원본 BGR 이미지
        pred_bin (np.ndarray): 전체 예측 binary 마스크 (인스턴스 union)
        alpha (float): 오버레이 투명도
    Returns:
        np.ndarray: 단일 색 오버레이 BGR 이미지
    """
    overlay = bgr.copy()
    color_layer = np.zeros_like(bgr)
    region = pred_bin.astype(bool)
    color_layer[region] = (0, 0, 255)   # 단일 빨강
    overlay[region] = (
        overlay[region] * (1 - alpha) + color_layer[region] * alpha
    ).astype(np.uint8)
    return overlay


# ---- 테스트셋 샘플 선택 ----

target_size = (RESIZE_TO, RESIZE_TO)

fns_all = sorted([
    fn for fn in os.listdir(TEST_IMG_DIR)
    if fn.lower().endswith((".png", ".jpg", ".jpeg"))
])

areas = []
gt_bin_cache = {}

for fn in fns_all:
    gt_path = os.path.join(TEST_MSK_DIR, fn)
    if not os.path.exists(gt_path):
        continue
    gt = read_gt_bin(gt_path, target_size=target_size)
    if gt is None or gt.sum() == 0:
        continue
    areas.append((fn, int(gt.sum())))
    gt_bin_cache[fn] = gt

areas.sort(key=lambda x: x[1])
K = min(NUM_TEST_SAMPLES, len(areas))
k_s, k_m = int(round(K * 0.30)), int(round(K * 0.40))
k_l = K - k_s - k_m
n   = len(areas)

test_fns = (
    [fn for fn, _ in areas[:k_s]] +
    [fn for fn, _ in areas[n // 2 - k_m // 2 : n // 2 - k_m // 2 + k_m]] +
    [fn for fn, _ in areas[-k_l:]]
)
test_fns = list(dict.fromkeys(test_fns))[:K]
print(f"테스트 샘플 수: {len(test_fns)}")

In [ ]:
# ---- 추론 실행 ----
# Mask R-CNN은 인스턴스 단위 출력 → 각 상처를 독립적으로 분리하여 저장
# (U-Net은 이미지 전체를 단일 마스크로 출력하므로 다중 상처 독립 추적 불가)

os.makedirs(INFER_OUT_DIR, exist_ok=True)
plot_dir = os.path.join(INFER_OUT_DIR, "plots")
os.makedirs(plot_dir, exist_ok=True)

model.eval()
model.to(device)

rows = []
t0   = time.time()

for idx, fn in enumerate(test_fns, 1):
    bgr = cv2.imread(os.path.join(TEST_IMG_DIR, fn), cv2.IMREAD_COLOR)
    if bgr is None:
        continue
    bgr = cv2.resize(bgr, target_size, interpolation=cv2.INTER_AREA)

    # BGR -> RGB tensor
    rgb   = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    img_t = torch.from_numpy(rgb).permute(2, 0, 1).float().div(255.0).to(device)

    # ---- 인스턴스별 마스크 분리 수집 ----
    instance_masks = []   # list[np.ndarray]: 인스턴스별 binary 마스크
    instance_areas = []   # list[int]:        인스턴스별 픽셀 면적
    pred_score     = None

    with torch.inference_mode():
        out = model([img_t])[0]

    scores_np  = out.get("scores", torch.tensor([])).detach().cpu().numpy()
    masks_tens = out.get("masks", None)

    if len(scores_np) > 0 and masks_tens is not None:
        pred_score = float(scores_np[0])

        # score 임계값 통과 + MAX_INSTANCES 개수 제한 (내림차순 정렬 유지)
        keep_idx = np.where(scores_np >= SCORE_THR)[0][:MAX_INSTANCES]

        for ki in keep_idx:
            inst_bin = (masks_tens[ki, 0].detach().cpu().numpy() > MASK_THR).astype(np.uint8)
            inst_bin = postprocess_mask(inst_bin)
            if inst_bin.sum() > 0:               # 빈 마스크 제외
                instance_masks.append(inst_bin)
                instance_areas.append(int(inst_bin.sum()))

    # 전체 예측 마스크: 인스턴스 union (평가 지표용)
    if instance_masks:
        pred_bin = np.stack(instance_masks).max(axis=0)
    else:
        pred_bin = np.zeros((RESIZE_TO, RESIZE_TO), dtype=np.uint8)

    num_instances = len(instance_masks)
    pred_area     = int(pred_bin.sum())
    gt_bin        = gt_bin_cache.get(fn)
    gt_area       = int(gt_bin.sum()) if gt_bin is not None else None

    # ---- 평가 지표 ----
    iou = dice = prec = rec = None
    tp  = fp = fn_ = abs_err = rel_err = None

    if gt_bin is not None and gt_area and gt_area > 0:
        iou, dice, prec, rec, tp, fp, fn_ = compute_metrics(pred_bin, gt_bin)
        abs_err = abs(pred_area - gt_area)
        rel_err = abs_err / gt_area

    # ---- 오버레이 저장 ----

    # 1) GT 비교 오버레이 (GT green / Pred red / Overlap yellow)
    overlay_path = None
    if gt_bin is not None:
        ov = make_overlay(bgr, pred_bin, gt_bin)
        overlay_path = os.path.join(INFER_OUT_DIR, f"overlay_{fn}")
        cv2.imwrite(overlay_path, ov)

    # 2) 다중 인스턴스 오버레이: 인스턴스별 고유 색상 (Mask R-CNN 장점 시각화)
    multi_overlay_path = None
    if num_instances >= 1:
        mo = make_multi_instance_overlay(bgr, instance_masks)
        multi_overlay_path = os.path.join(INFER_OUT_DIR, f"multi_{fn}")
        cv2.imwrite(multi_overlay_path, mo)

    # 3) 병합 오버레이: 단일 색(빨강) — U-Net 방식 시뮬레이션
    merged_overlay_path = None
    if pred_area > 0:
        mgo = make_merged_overlay(bgr, pred_bin)
        merged_overlay_path = os.path.join(INFER_OUT_DIR, f"merged_{fn}")
        cv2.imwrite(merged_overlay_path, mgo)

    rows.append(dict(
        filename=fn,
        pred_score_top1=pred_score,
        num_instances=num_instances,
        instance_areas=str(instance_areas),      # ex) "[12045, 3821]"
        pred_area_px=pred_area,
        gt_area_px=gt_area,
        abs_err_px=abs_err,
        rel_err=rel_err,
        iou=iou, dice=dice, precision=prec, recall=rec,
        tp=tp, fp=fp, fn=fn_,
        overlay_path=overlay_path,
        multi_overlay_path=multi_overlay_path,
        merged_overlay_path=merged_overlay_path,
    ))

    if idx % 20 == 0 or idx == len(test_fns):
        multi_cnt = sum(1 for r in rows if r["num_instances"] >= 2)
        print(f"[{idx}/{len(test_fns)}] elapsed={time.time()-t0:.1f}s | "
              f"다중 인스턴스: {multi_cnt}개")

df = pd.DataFrame(rows)
csv_path = os.path.join(INFER_OUT_DIR, "infer_results.csv")
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
print("CSV 저장 ->", csv_path)
print(f"\n[인스턴스 분포]")
print(df["num_instances"].value_counts().sort_index().to_string())

## 8. 결과 요약 & 시각화

In [ ]:
valid = df.dropna(subset=["iou", "dice", "precision", "recall", "rel_err"])

summary = {
    "num_selected"       : len(df),
    "num_valid_gt"       : len(valid),
    "score_thr"          : SCORE_THR,
    "mask_thr"           : MASK_THR,
    "mean_iou"           : valid["iou"].mean()       if len(valid) else None,
    "mean_dice"          : valid["dice"].mean()      if len(valid) else None,
    "mean_precision"     : valid["precision"].mean() if len(valid) else None,
    "mean_recall"        : valid["recall"].mean()    if len(valid) else None,
    "MAE_abs_err_px"     : valid["abs_err_px"].mean()if len(valid) else None,
    "MAPE_rel_err"       : valid["rel_err"].mean()   if len(valid) else None,
    "pred_area_zero_rate": (df["pred_area_px"] == 0).mean(),
    "train_sample_size"  : TRAIN_SAMPLE_SIZE,
    "epochs"             : EPOCHS,
    "freeze_backbone"    : FREEZE_BACKBONE,
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv(os.path.join(INFER_OUT_DIR, "summary.csv"),
                  index=False, encoding="utf-8-sig")

print("===== 평가 결과 =====")
for k, v in summary.items():
    print(f"  {k:25s}: {v}")

# ---- Plots ----

# 1) Pred area vs GT area scatter
plt.figure(figsize=(6, 5))
sub = df.dropna(subset=["gt_area_px"])
plt.scatter(sub["gt_area_px"], sub["pred_area_px"], alpha=0.6)
lim = max(sub["gt_area_px"].max(), sub["pred_area_px"].max()) * 1.05
plt.plot([0, lim], [0, lim], "r--", label="y=x (perfect)")
plt.xlabel("GT area (px)")
plt.ylabel("Pred area (px)")
plt.title("Pred vs GT Area")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(plot_dir, "area_scatter.png"), dpi=150)
plt.show()

# 2) IoU histogram
plt.figure(figsize=(6, 4))
plt.hist(valid["iou"].dropna().values, bins=20, edgecolor="black")
plt.axvline(valid["iou"].mean(), color="red", linestyle="--",
            label=f"mean={valid['iou'].mean():.3f}")
plt.xlabel("IoU")
plt.ylabel("Count")
plt.title("IoU Distribution")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(plot_dir, "iou_hist.png"), dpi=150)
plt.show()

# 3) Training loss curve
plt.figure(figsize=(6, 4))
plt.plot(range(1, EPOCHS + 1), loss_history, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Avg Loss")
plt.title("Training Loss")
plt.tight_layout()
plt.savefig(os.path.join(plot_dir, "train_loss.png"), dpi=150)
plt.show()

print("모든 결과 저장 완료 ->", INFER_OUT_DIR)

## 10. 다중 상처 인스턴스 시연 — Mask R-CNN vs U-Net

### 왜 Mask R-CNN인가
당뇨발·화상·욕창 환자는 한 이미지에 여러 상처가 동시에 존재한다.

| | Semantic Seg (U-Net) | Instance Seg (Mask R-CNN) |
|---|---|---|
| 다중 상처 | 하나의 덩어리로 합침 | 각 상처를 #1, #2, ... 로 분리 |
| 인스턴스별 RVI | ❌ 불가 | ✅ 상처마다 독립 산출 |

아래에서 같은 이미지를 두 방식으로 시각화하여 차이를 확인한다.

In [ ]:
# ---- 다중 상처 인스턴스 시연 ----

# 다중 인스턴스(2개+) 이미지 우선 선택, 없으면 단일 인스턴스로 대체
multi_df = df[df["num_instances"] >= 2].copy()
print(f"다중 인스턴스(2개+) 검출 이미지: {len(multi_df)}개")

if len(multi_df) == 0:
    # 다중 상처가 없어도 개념 시연 가능: 단일 인스턴스 이미지 사용
    multi_df = df[df["num_instances"] >= 1].head(4).copy()
    title_note = "(단일 인스턴스 이미지로 개념 시연)"
    print(f"→ {title_note}")
else:
    title_note = ""

show_df = multi_df.head(4)
n_rows  = len(show_df)

fig, axes = plt.subplots(n_rows, 3, figsize=(15, 5 * n_rows))
if n_rows == 1:
    axes = axes[np.newaxis, :]

for row_i, (_, row) in enumerate(show_df.iterrows()):
    fn = row["filename"]

    # 원본 이미지 로드
    bgr = cv2.imread(os.path.join(TEST_IMG_DIR, fn), cv2.IMREAD_COLOR)
    if bgr is None:
        continue
    bgr = cv2.resize(bgr, target_size, interpolation=cv2.INTER_AREA)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    # 다중 인스턴스 오버레이 (Mask R-CNN)
    mo_path = str(row.get("multi_overlay_path", ""))
    mo_bgr  = cv2.imread(mo_path) if os.path.exists(mo_path) else bgr
    mo_rgb  = cv2.cvtColor(mo_bgr, cv2.COLOR_BGR2RGB)

    # 병합 오버레이 (U-Net 시뮬레이션)
    mg_path = str(row.get("merged_overlay_path", ""))
    mg_bgr  = cv2.imread(mg_path) if os.path.exists(mg_path) else bgr
    mg_rgb  = cv2.cvtColor(mg_bgr, cv2.COLOR_BGR2RGB)

    n_inst     = int(row["num_instances"])
    areas_str  = row["instance_areas"]
    iou_val    = f"{row['iou']:.3f}" if pd.notna(row.get("iou")) else "N/A"

    # --- Col 0: 원본 ---
    axes[row_i, 0].imshow(rgb)
    axes[row_i, 0].set_title(f"원본 이미지\n{fn[:25]}", fontsize=10)
    axes[row_i, 0].axis("off")

    # --- Col 1: Mask R-CNN (인스턴스별 분리) ---
    axes[row_i, 1].imshow(mo_rgb)
    axes[row_i, 1].set_title(
        f"Mask R-CNN — {n_inst}개 인스턴스 분리\n"
        f"면적: {areas_str}\n(각 상처 독립 RVI 산출 가능)",
        fontsize=9
    )
    axes[row_i, 1].axis("off")

    # --- Col 2: U-Net 방식 시뮬레이션 (단일 합산 마스크) ---
    axes[row_i, 2].imshow(mg_rgb)
    axes[row_i, 2].set_title(
        f"Semantic Seg (U-Net 방식)\n"
        f"모든 인스턴스 → 단일 덩어리\n(인스턴스 구분 & 개별 추적 불가)",
        fontsize=9
    )
    axes[row_i, 2].axis("off")

plt.suptitle(
    f"Mask R-CNN vs Semantic Segmentation\n"
    f"다중 상처 인스턴스 분리 능력 비교 {title_note}",
    fontsize=13, y=1.01
)
plt.tight_layout()

demo_path = os.path.join(plot_dir, "multi_instance_demo.png")
plt.savefig(demo_path, dpi=150, bbox_inches="tight")
plt.show()
print("저장 ->", demo_path)

## 11. 합성 다중 상처 이미지로 인스턴스 분리 시연

현재 데이터셋은 이미지당 상처 1개 기준으로 수집되었다.
다중 상처 시나리오를 시연하기 위해 **두 이미지의 상처 영역을 합성**한다.

### 합성 방법
1. IoU 상위 이미지 A를 베이스 이미지로 선택
2. 다른 이미지 B의 상처 영역을 마스크 기반으로 추출
3. A의 비어있는 위치에 B의 상처를 붙여넣음
4. 합성 이미지에 모델 추론 → 인스턴스 분리 확인

> **주의**: 학습 데이터에 다중 상처 이미지가 없어 검출 성능이 보장되지 않지만,  
> Mask R-CNN의 구조적 설계(RPN + RoIAlign)는 다중 인스턴스를 지원한다.

In [ ]:
# ---- 합성 다중 상처 이미지 생성 및 인스턴스 분리 시연 ----

def create_synthetic_multi_wound(base_img: np.ndarray,
                                  base_mask: np.ndarray,
                                  donor_img: np.ndarray,
                                  donor_mask: np.ndarray,
                                  margin: int = 20) -> tuple:
    """
    베이스 이미지에 기증 이미지의 상처 영역을 합성하여 다중 상처 이미지를 생성.

    Args:
        base_img (np.ndarray): 베이스 BGR 이미지 (512x512)
        base_mask (np.ndarray): 베이스 상처 마스크
        donor_img (np.ndarray): 기증 BGR 이미지 (512x512)
        donor_mask (np.ndarray): 기증 상처 마스크
        margin (int): 상처 bbox 주변 여백
    Returns:
        tuple: (합성 이미지, 합성 마스크, 붙여넣기 위치 정보)
    """
    H, W = base_img.shape[:2]

    # donor 상처 bbox 추출
    ys, xs = np.where(donor_mask > 0)
    if len(xs) == 0:
        return base_img.copy(), base_mask.copy(), None

    y1, y2 = max(ys.min() - margin, 0), min(ys.max() + margin, H)
    x1, x2 = max(xs.min() - margin, 0), min(xs.max() + margin, W)
    crop_h, crop_w = y2 - y1, x2 - x1

    wound_crop = donor_img[y1:y2, x1:x2]
    mask_crop  = donor_mask[y1:y2, x1:x2]

    # base 상처와 겹치지 않는 위치 찾기 (반대 사분면 우선)
    base_cy = int(np.where(base_mask > 0)[0].mean()) if base_mask.sum() > 0 else H // 2
    base_cx = int(np.where(base_mask > 0)[1].mean()) if base_mask.sum() > 0 else W // 2

    # base 상처 중심의 반대 사분면에 배치
    if base_cx < W // 2:
        target_x = min(W - crop_w - 10, W * 3 // 4 - crop_w // 2)
    else:
        target_x = max(10, W // 4 - crop_w // 2)

    if base_cy < H // 2:
        target_y = min(H - crop_h - 10, H * 3 // 4 - crop_h // 2)
    else:
        target_y = max(10, H // 4 - crop_h // 2)

    target_x = max(0, min(target_x, W - crop_w))
    target_y = max(0, min(target_y, H - crop_h))

    # 합성: 마스크 영역만 덧씌우기 (자연스러운 합성)
    synth_img  = base_img.copy()
    synth_mask = base_mask.copy()

    region_mask = mask_crop.astype(bool)
    for c in range(3):
        synth_img[target_y:target_y+crop_h, target_x:target_x+crop_w][region_mask, c] = \
            wound_crop[region_mask, c]

    synth_mask[target_y:target_y+crop_h, target_x:target_x+crop_w][region_mask] = 1

    info = dict(target_x=target_x, target_y=target_y,
                crop_w=crop_w, crop_h=crop_h)
    return synth_img, synth_mask, info


def run_inference_single(model, bgr_img: np.ndarray,
                          score_thr: float, mask_thr: float,
                          max_inst: int) -> list:
    """
    단일 이미지에 대해 추론을 수행하고 인스턴스별 binary 마스크 리스트를 반환.

    Args:
        model: Mask R-CNN 모델
        bgr_img (np.ndarray): BGR 이미지
        score_thr (float): score 임계값
        mask_thr (float): mask 임계값
        max_inst (int): 최대 인스턴스 수
    Returns:
        list[np.ndarray]: 인스턴스별 binary 마스크
    """
    rgb   = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2RGB)
    img_t = torch.from_numpy(rgb).permute(2, 0, 1).float().div(255.0).to(device)

    model.eval()
    with torch.inference_mode():
        out = model([img_t])[0]

    scores_np  = out.get("scores", torch.tensor([])).detach().cpu().numpy()
    masks_tens = out.get("masks", None)

    instances = []
    if len(scores_np) > 0 and masks_tens is not None:
        keep_idx = np.where(scores_np >= score_thr)[0][:max_inst]
        for ki in keep_idx:
            m = (masks_tens[ki, 0].detach().cpu().numpy() > mask_thr).astype(np.uint8)
            m = postprocess_mask(m)
            if m.sum() > 0:
                instances.append(m)
    return instances


# ---- 합성 쌍 선택: IoU 상위 이미지 중 공간적으로 다른 상처 위치 조합 ----

valid_df = df.dropna(subset=["iou"]).sort_values("iou", ascending=False)

# 상처 위치(중심)가 다른 두 이미지를 골라 합성
synth_pairs = []
used = set()

for _, row_a in valid_df.iterrows():
    if len(synth_pairs) >= 3:
        break
    fn_a = row_a["filename"]
    if fn_a in used:
        continue

    gt_a = gt_bin_cache.get(fn_a)
    if gt_a is None or gt_a.sum() == 0:
        continue
    cy_a = int(np.where(gt_a > 0)[0].mean())
    cx_a = int(np.where(gt_a > 0)[1].mean())

    for _, row_b in valid_df.iterrows():
        fn_b = row_b["filename"]
        if fn_b == fn_a or fn_b in used:
            continue
        gt_b = gt_bin_cache.get(fn_b)
        if gt_b is None or gt_b.sum() == 0:
            continue
        cy_b = int(np.where(gt_b > 0)[0].mean())
        cx_b = int(np.where(gt_b > 0)[1].mean())

        # 두 상처 중심이 충분히 떨어진 쌍 선택 (거리 > 150px)
        dist = np.sqrt((cx_a - cx_b) ** 2 + (cy_a - cy_b) ** 2)
        if dist > 150:
            synth_pairs.append((fn_a, fn_b))
            used.add(fn_a)
            used.add(fn_b)
            break

print(f"합성 쌍 수: {len(synth_pairs)}")
for a, b in synth_pairs:
    print(f"  {a}  +  {b}")

# ---- 합성 → 추론 → 시각화 ----

synth_out_dir = os.path.join(INFER_OUT_DIR, "synthetic_multi")
os.makedirs(synth_out_dir, exist_ok=True)

n_pairs = len(synth_pairs)
if n_pairs == 0:
    print("합성 가능한 쌍이 없습니다. valid_df 크기:", len(valid_df))
else:
    fig, axes = plt.subplots(n_pairs, 3, figsize=(15, 5 * n_pairs))
    if n_pairs == 1:
        axes = axes[np.newaxis, :]

    for row_i, (fn_a, fn_b) in enumerate(synth_pairs):
        # 이미지 & 마스크 로드
        bgr_a = cv2.resize(
            cv2.imread(os.path.join(TEST_IMG_DIR, fn_a)), target_size)
        bgr_b = cv2.resize(
            cv2.imread(os.path.join(TEST_IMG_DIR, fn_b)), target_size)
        gt_a  = gt_bin_cache[fn_a]
        gt_b  = gt_bin_cache[fn_b]

        # 합성
        synth_bgr, synth_mask, info = create_synthetic_multi_wound(
            bgr_a, gt_a, bgr_b, gt_b)

        # 합성 이미지 저장
        synth_path = os.path.join(synth_out_dir, f"synth_{row_i}.png")
        cv2.imwrite(synth_path, synth_bgr)

        # 추론
        instances = run_inference_single(
            model, synth_bgr, SCORE_THR, MASK_THR, MAX_INSTANCES)

        print(f"[쌍 {row_i+1}] 검출 인스턴스: {len(instances)}개 "
              f"| 면적: {[int(m.sum()) for m in instances]}")

        # 시각화용 이미지 준비
        synth_rgb = cv2.cvtColor(synth_bgr, cv2.COLOR_BGR2RGB)

        # Mask R-CNN: 인스턴스별 색상
        mo = make_multi_instance_overlay(synth_bgr, instances) if instances \
             else synth_bgr.copy()
        mo_rgb = cv2.cvtColor(mo, cv2.COLOR_BGR2RGB)

        # U-Net 시뮬레이션: 모든 인스턴스 합산
        if instances:
            merged = np.stack(instances).max(axis=0)
        else:
            merged = np.zeros((RESIZE_TO, RESIZE_TO), dtype=np.uint8)
        mg = make_merged_overlay(synth_bgr, merged)
        mg_rgb = cv2.cvtColor(mg, cv2.COLOR_BGR2RGB)

        # --- 3열 플롯 ---
        axes[row_i, 0].imshow(synth_rgb)
        axes[row_i, 0].set_title(
            f"합성 이미지\n(상처 A: {fn_a[:18]}\n+ 상처 B: {fn_b[:18]})",
            fontsize=8)
        axes[row_i, 0].axis("off")

        axes[row_i, 1].imshow(mo_rgb)
        n_inst = len(instances)
        areas  = [int(m.sum()) for m in instances]
        axes[row_i, 1].set_title(
            f"Mask R-CNN — {n_inst}개 인스턴스 분리\n"
            f"면적: {areas}\n→ 상처별 독립 RVI 산출 가능",
            fontsize=9)
        axes[row_i, 1].axis("off")

        axes[row_i, 2].imshow(mg_rgb)
        axes[row_i, 2].set_title(
            "Semantic Seg (U-Net 방식)\n"
            "모든 상처 → 단일 덩어리\n→ 인스턴스 구분 불가",
            fontsize=9)
        axes[row_i, 2].axis("off")

    plt.suptitle(
        "합성 다중 상처 이미지: Mask R-CNN vs Semantic Segmentation\n"
        "(Mask R-CNN 선택 근거 — 인스턴스 단위 추적)",
        fontsize=13, y=1.01)
    plt.tight_layout()

    demo_synth_path = os.path.join(plot_dir, "synthetic_multi_wound_demo.png")
    plt.savefig(demo_synth_path, dpi=150, bbox_inches="tight")
    plt.show()
    print("저장 ->", demo_synth_path)